In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window as W
 
silver_path = "s3://my-iot-data-bucket-2025/silver"

silver_df = spark.read.format("delta").load(silver_path + "/iot")

silver = (silver_df
          .withColumn("timestamp", F.col("timestamp").cast("timestamp"))
          .withColumn("is_faulty", F.col("is_faulty").cast("boolean"))
         )

# 1) Per-device (and optionally per day) ordering
#    If you want daily MTTR, include "event_date" in partition keys below.
w_dev = W.partitionBy("device_id").orderBy("timestamp")

# 2) Detect state changes (healthy <-> faulty)
prev_faulty = F.lag("is_faulty").over(w_dev)
silver = silver.withColumn(
    "status_change",
    F.when(prev_faulty.isNull() | (F.col("is_faulty") != prev_faulty), F.lit(1)).otherwise(F.lit(0))
)

# 3) Build a run_id that increments whenever status changes
silver = silver.withColumn("run_id", F.sum("status_change").over(w_dev.rowsBetween(W.unboundedPreceding, 0)))

# 4) Collapse to one row per continuous run
runs = (silver
    .groupBy("device_id", "run_id")
    .agg(
        F.first("is_faulty").alias("run_is_faulty"),
        F.min("timestamp").alias("run_start_ts"),
        F.max("timestamp").alias("run_end_ts")  # not used for MTTR end, but handy for QA
    )
)

w_runs = W.partitionBy("device_id").orderBy("run_id")
runs = (runs
    .withColumn("next_run_is_faulty", F.lead("run_is_faulty").over(w_runs))
    .withColumn("next_run_start_ts", F.lead("run_start_ts").over(w_runs))
)

# 6) Observation end (used if a device never recovers after a fault)
obs_end = silver.groupBy("device_id").agg(F.max("timestamp").alias("obs_end_ts"))

runs = runs.join(obs_end, "device_id", "left")

# Select necessary columns to avoid ambiguity
runs = runs.select(
    "device_id",
    "run_is_faulty",
    "next_run_is_faulty",
    "next_run_start_ts",
    "run_start_ts",
    "obs_end_ts"
)

# 7) Keep only faulty runs; repair end is first healthy start after the run
fault_runs = (runs
    .filter(F.col("run_is_faulty") == True)
    .withColumn(
        "repair_end_ts",
        F.when(F.col("next_run_is_faulty") == False, F.col("next_run_start_ts"))  # next run is healthy
         .otherwise(F.col("obs_end_ts"))  # never recovers within window
    )
    .filter(F.col("repair_end_ts").isNotNull())  # guard against completely empty future
)

# 8)
fault_runs = fault_runs.withColumn(
    "repair_seconds",
    F.col("repair_end_ts").cast("long") - F.col("run_start_ts").cast("long")
)

mttr_device = (fault_runs
    .groupBy("device_id")
    .agg(F.round(F.avg("repair_seconds"), 2).alias("mttr_seconds"))
    .withColumn("mttr_minutes", F.round((F.col("mttr_seconds")/60.0), 2))
)

# 5)

In [0]:
# from pyspark.sql.functions import avg, sum, count, when

battery = silver_df.groupBy("device_id").agg(
    F.round(F.avg("battery"), 2).alias("battery"),
    F.round(
        (F.sum(F.when(silver_df["battery"] < 30, 1).otherwise(0)) / F.count("battery") * 100.0), 
        2
    ).alias("low_battery_percent")
)


In [0]:
fault_frequency = silver_df.filter(silver_df["is_faulty"] == True) \
                    .withColumn("week", F.weekofyear("timestamp")) \
                    .groupBy("device_id", "week") \
                    .agg(F.count("is_faulty").alias("fault_count"))


In [0]:
from pyspark.sql import functions as F

joined_df = (
    silver_df.join(mttr_device, "device_id")
    .select(
        "device_id",
        "event_date",
        "mttr_seconds",
        "mttr_minutes"
    )
)

final_df = (
    joined_df.alias("jd")
    .join(battery.alias("b"), "device_id", "left")
    .join(
        fault_frequency.alias("ff"),
        (joined_df["device_id"] == fault_frequency["device_id"]) &
        (F.weekofyear(joined_df["event_date"]) == fault_frequency["week"]),
        "left"
    )
    .select(
        "jd.device_id",
        "jd.event_date",
        "jd.mttr_seconds",
        "jd.mttr_minutes",
        F.col("b.battery").alias("avg_battery"),
        "b.low_battery_percent",
        "ff.week",
        "ff.fault_count"
    )
)


In [0]:
gold_path = "s3://my-iot-data-bucket-2025/gold"
final_df.display()
final_df.write.format("delta").mode("overwrite").option("mergeSchema", "true").partitionBy("event_date").save(gold_path + "/mttr_kpi")

# Explanation for visualizations:
# 1. MTTR (Mean Time to Repair) per device: This visualization shows the average time it takes to repair a device after a fault.
# 2. Average Battery Level per device: This visualization displays the average battery level for each device.
# 3. Low Battery Percentage per device: This visualization indicates the percentage of time a device's battery level was below 30%.
# 4. Fault Frequency per device per week: This visualization shows the number of faults each device experienced per week.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.